# Post-Tuning Evaluation

This notebook evaluates the fine-tuned MiniCPM phonetic evaluator against the baseline ASR results. The baseline ASR marks only exact transcript matches as correct; the tuned SLM is used as a second-stage judge for strict-match failures, where child speech patterns, lisps, plurals, or near-phonetic transcripts may still be acceptable.

In [1]:
from pathlib import Path

import modal
import pandas as pd

BASELINE_RESULTS = Path("data/baseline_results.csv")
if not BASELINE_RESULTS.exists():
    BASELINE_RESULTS = Path("../data/baseline_results.csv")
MODAL_APP_NAME = "read-along-ai-inference"
MODAL_FUNCTION_NAME = "run_minicpm_evaluator"

df = pd.read_csv(BASELINE_RESULTS)
df["Strict Match"] = df["Strict Match"].astype(bool)
df.head()

,File,Target Word,ASR Transcript,Strict Match
0,data/processed_audio/jane_cleaned/01_scientist...,scientist,scientists,False
1,data/processed_audio/jane_cleaned/02_safari.wav,safari,so far,False
2,data/processed_audio/jane_cleaned/03_secret.wav,secret,secret,True
3,data/processed_audio/jane_cleaned/04_sunflower...,sunflower,sunny flowers,False
4,data/processed_audio/jane_cleaned/05_system.wav,system,system,True


In [2]:
failures = df.loc[~df["Strict Match"]].copy()
print(f"Strict-match failures to review with MiniCPM: {len(failures)}")
failures[["File", "Target Word", "ASR Transcript", "Strict Match"]]

Strict-match failures to review with MiniCPM: 12


,File,Target Word,ASR Transcript,Strict Match
0,data/processed_audio/jane_cleaned/01_scientist...,scientist,scientists,False
1,data/processed_audio/jane_cleaned/02_safari.wav,safari,so far,False
3,data/processed_audio/jane_cleaned/04_sunflower...,sunflower,sunny flowers,False
7,data/processed_audio/jane_cleaned/08_sudden.wav,sudden,seven,False
11,data/processed_audio/jane_cleaned/12_invisible...,invisible,and where's the ball,False
12,data/processed_audio/jane_cleaned/13_recipe.wav,recipe,this is tea,False
13,data/processed_audio/jane_cleaned/14_fossil.wav,fossil,five,False
15,data/processed_audio/jane_cleaned/16_massive.wav,massive,maze,False
17,data/processed_audio/jane_cleaned/18_guessing.wav,guessing,yeah same,False
28,data/processed_audio/jane_cleaned/29_compass.wav,compass,come this,False


## MiniCPM As A Phonetic Judge

The tuned model receives only the target word and ASR transcript, then outputs `True` or `False`. A `True` verdict means the transcript is an acceptable phonetic match for the intended word, even if the baseline ASR transcript was not an exact string match.

In [3]:
def modal_function(app_name: str, function_name: str):
    lookup = getattr(modal.Function, "lookup", None)
    if lookup is not None:
        return lookup(app_name, function_name)
    return modal.Function.from_name(app_name, function_name)


evaluator = modal_function(MODAL_APP_NAME, MODAL_FUNCTION_NAME)
slm_verdicts = []

for _, row in failures.iterrows():
    verdict = evaluator.remote(row["Target Word"], row["ASR Transcript"])
    slm_verdicts.append(str(verdict).strip())

failures["SLM Verdict"] = slm_verdicts
failures["SLM Accepts"] = failures["SLM Verdict"].str.casefold().eq("true")
failures[["Target Word", "ASR Transcript", "SLM Verdict", "SLM Accepts"]]

,Target Word,ASR Transcript,SLM Verdict,SLM Accepts
0,scientist,scientists,True,True
1,safari,so far,False,False
3,sunflower,sunny flowers,True,True
7,sudden,seven,False,False
11,invisible,and where's the ball,False,False
12,recipe,this is tea,False,False
13,fossil,five,False,False
15,massive,maze,False,False
17,guessing,yeah same,True,True
28,compass,come this,False,False


In [4]:
baseline_correct = int(df["Strict Match"].sum())
slm_recovered = int(failures["SLM Accepts"].sum())
new_correct = baseline_correct + slm_recovered
total = len(df)

summary = pd.DataFrame(
    [
        {
            "System": "Baseline ASR exact match",
            "Correct": baseline_correct,
            "Total": total,
            "Accuracy": baseline_correct / total,
        },
        {
            "System": "ASR + tuned MiniCPM phonetic judge",
            "Correct": new_correct,
            "Total": total,
            "Accuracy": new_correct / total,
        },
    ]
)

comparison = failures.assign(
    **{
        "Baseline Verdict": "False",
        "New System Verdict": failures["SLM Verdict"],
    }
)[["Target Word", "ASR Transcript", "Baseline Verdict", "New System Verdict"]]

display(
    summary.style.format({"Accuracy": "{:.1%}"}).hide(axis="index")
)
display(
    comparison.style.set_caption("Strict-match failures reviewed by tuned MiniCPM")
)

System,Correct,Total,Accuracy
Baseline ASR exact match,38,50,76.0%
ASR + tuned MiniCPM phonetic judge,42,50,84.0%


,Target Word,ASR Transcript,Baseline Verdict,New System Verdict
0,scientist,scientists,False,True
1,safari,so far,False,False
3,sunflower,sunny flowers,False,True
7,sudden,seven,False,False
11,invisible,and where's the ball,False,False
12,recipe,this is tea,False,False
13,fossil,five,False,False
15,massive,maze,False,False
17,guessing,yeah same,False,True
28,compass,come this,False,False
